In [1]:
from pathlib import Path
import pandas as pd

In [2]:
saved_dir = Path(r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\saved_data\ROAD")

train_dfs = pd.read_pickle(saved_dir / "train_dfs.pkl")
test_dfs = pd.read_pickle(saved_dir / "test_dfs.pkl")

print("Reloaded train_dfs and test_dfs from saved_data/")
print("Train dfs:", len(train_dfs))
print("Test dfs:", len(test_dfs))
print("Train sizes:", [len(df) for df in train_dfs])
print("Test sizes:", [len(df) for df in test_dfs])

Reloaded train_dfs and test_dfs from saved_data/
Train dfs: 38
Test dfs: 38
Train sizes: [41580, 39911, 42657, 40945, 25197, 24186, 8058, 8024, 49098, 47143, 62576, 60064, 121688, 116802, 13400, 12862, 47249, 45352, 48501, 46554, 39720, 38128, 73526, 70575, 46872, 44992, 2393018, 850900, 617216, 1139367, 632354, 746282, 91319, 3535538, 1257855, 98438, 899801, 7057533]
Test sizes: [10394, 9977, 10664, 10236, 6299, 6046, 2014, 2006, 12274, 11785, 15644, 15016, 30421, 29200, 3349, 3215, 11812, 11338, 12125, 11638, 9930, 9531, 18381, 17643, 11718, 11247, 598254, 212724, 154303, 284841, 158088, 186570, 22829, 883884, 314463, 24609, 224950, 1764383]


In [3]:
train_dfs = pd.concat(train_dfs, ignore_index=True)
test_dfs = pd.concat(test_dfs, ignore_index=True)

print("Merged train_dfs shape:", train_dfs.shape)
print("Merged test_dfs shape:", test_dfs.shape)

Merged train_dfs shape: (20535281, 5)
Merged test_dfs shape: (5133801, 5)


In [4]:
#Calculate Data Length Code (DLC) - number of bytes in the Data field
train_dfs["DLC"] = train_dfs["Data"].str.replace(" ", "").str.len() // 2
test_dfs["DLC"] = test_dfs["Data"].str.replace(" ", "").str.len() // 2

print("Train DLC added:")
print(train_dfs[["Data", "DLC"]].head())
print("\nTest DLC added:")
print(test_dfs[["Data", "DLC"]].head())

Train DLC added:
               Data  DLC
0  8935000B09FFEC80    8
1  0000000000000000    8
2  0A770460F1030700    8
3  000650000C4317D0    8
4  900040DF403F6F60    8

Test DLC added:
               Data  DLC
0  595945450000FFFF    8
1  6000000000000000    8
2  000803C3EA11F4CE    8
3  0010FA24C12E10A0    8
4  00000000001D0800    8


In [5]:
import pandas as pd, numpy as np

# Only keep the columns you need
X_train = train_dfs.loc[:, ["ID", "DLC", "Data"]].copy()

# Convert hexadecimal CAN IDs directly to uint32
X_train["ID"] = np.fromiter(
    (int(x, 16) for x in X_train["ID"]),
    dtype=np.uint16,
    count=len(X_train)
)

# DLC only needs values 0-8
X_train["DLC"] = X_train["DLC"].astype("uint8")

# Convert payload into a 64-bit integer
X_train["Data"] = (
    X_train["Data"]
    .str.replace(" ", "", regex=False)
    .map(lambda x: int(x, 16))
    .astype("uint64")
)

# Labels
y_train = train_dfs["Attack"][:].astype("category")

print(X_train)

            ID  DLC                  Data
0         1505    8   9886808604374199424
1          651    8                     0
2          208    8    754076277014726400
3           51    8      1776810996209616
4          293    8  10376364869061406560
...        ...  ...                   ...
20535276   192    8   6917529027641081856
20535277  1644    8     36046389741879328
20535278  1176    8   9799762407369736198
20535279  4095    8                     0
20535280   813    8   9223376599922049024

[20535281 rows x 3 columns]


In [6]:
import pandas as pd

# Only keep the columns you need
X_test = test_dfs.loc[:, ["ID", "DLC", "Data"]].copy()

# Convert hexadecimal CAN IDs directly to uint32
X_test["ID"] = np.fromiter(
    (int(x, 16) for x in X_test["ID"]),
    dtype=np.uint16,
    count=len(X_test)
)

# DLC only needs values 0-8
X_test["DLC"] = X_test["DLC"].astype("uint8")

# Convert payload into a 64-bit integer
X_test["Data"] = (
    X_test["Data"]
    .str.replace(" ", "", regex=False)
    .map(lambda x: int(x, 16))
    .astype("uint64")
)

# Labels
y_test = test_dfs["Attack"][:].astype("category")

print(X_test)

           ID  DLC                 Data
0        1760    8  6438253304957960191
1         192    8  6917529027641081856
2         354    8     2255939794236622
3         167    8     4778635394158752
4          58    8              1902592
...       ...  ...                  ...
5133796   737    8                    4
5133797   852    8  2354115967719976192
5133798   339    8               786442
5133799  1505    8  9887969671473330304
5133800  1634    8  5386305155408855040

[5133801 rows x 3 columns]


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=10,
    random_state=42
)

rf.fit(X_train, y_train)

# Predict
predictions = rf.predict(X_test)

In [8]:
predictions
y_test

0          T
1          R
2          R
3          R
4          R
          ..
5133796    R
5133797    R
5133798    R
5133799    R
5133800    R
Name: Attack, Length: 5133801, dtype: category
Categories (2, str): ['R', 'T']

In [9]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9982


In [10]:
import joblib
hello()

joblib.dump(rf, "saved_models/random_forest_original_2.joblib")

NameError: name 'hello' is not defined

In [ ]:
rf_loaded = joblib.load("saved_models/random_forest_original_2.joblib")
print(rf_loaded)

RandomForestClassifier(n_estimators=10, random_state=42)


In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, average="weighted", zero_division=0)
recall = recall_score(y_test, predictions, average="weighted", zero_division=0)
f1 = f1_score(y_test, predictions, average="weighted", zero_division=0)

labels = list(rf.classes_)
cm = confusion_matrix(y_test, predictions, labels=labels)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)

fpr = {}
fnr = {}
for i, label in enumerate(labels):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    fpr[label] = fp / (fp + tn) if (fp + tn) != 0 else 0.0
    fnr[label] = fn / (fn + tp) if (fn + tp) != 0 else 0.0

print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted precision: {precision:.4f}")
print(f"Weighted recall: {recall:.4f}")
print(f"Weighted F1 score: {f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, predictions, labels=labels, zero_division=0))
print("Confusion matrix:")
print(cm_df)
print("\nFalse positive rate per class:")
for label, rate in fpr.items():
    print(f"  {label}: {rate:.4f}")
print("\nFalse negative rate per class:")
for label, rate in fnr.items():
    print(f"  {label}: {rate:.4f}")

Accuracy: 0.9982
Weighted precision: 0.9979
Weighted recall: 0.9982
Weighted F1 score: 0.9980

Classification report:
              precision    recall  f1-score   support

           R       1.00      1.00      1.00   5121499
           T       0.71      0.43      0.54     12302

    accuracy                           1.00   5133801
   macro avg       0.85      0.72      0.77   5133801
weighted avg       1.00      1.00      1.00   5133801

Confusion matrix:
         R     T
R  5119323  2176
T     7006  5296

False positive rate per class:
  R: 0.5695
  T: 0.0004

False negative rate per class:
  R: 0.0004
  T: 0.5695
